# 14i -- Track-Quality Pre-Filter Sweep

CPU-only sweep over average detection confidence and minimum tracklet length using the saved 14h v3 Stage 2 outputs.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import tarfile
import time
from dataclasses import replace
from datetime import datetime
from pathlib import Path

import numpy as np

WORK_DIR = Path("/kaggle/working")
PROJECT = WORK_DIR / "gp"
INPUT_ROOT = Path("/kaggle/input")
DATA_OUT = Path("/tmp/pipeline_outputs")
DATA_OUT.mkdir(parents=True, exist_ok=True)

print(f"Python: {sys.version.split()[0]}")
print(f"Kaggle input exists: {INPUT_ROOT.exists()}")

## 1. Clone Repo And Install CPU Dependencies

In [ ]:
REPO_URL = "https://github.com/MRKDaGods/gp.git"

if not PROJECT.exists():
    print(f"Cloning {REPO_URL} ...")
    subprocess.check_call(["git", "clone", "--depth", "1", "-b", "feature/pretrained-ensemble", REPO_URL, str(PROJECT)])
else:
    print("Repo already present; pulling latest ...")
    subprocess.check_call(["git", "-C", str(PROJECT), "pull", "--ff-only"])

os.chdir(str(PROJECT))
sys.path.insert(0, str(PROJECT))

LOCAL_14I_FILES = {
    "scripts/filter_tracklets.py": "\"\"\"Filter Stage 1/2 tracklet artifacts by track length and confidence.\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\n\nfrom src.core.io_utils import load_tracklets_by_camera, save_tracklets_by_camera\n\n\nOPTIONAL_STAGE2_ARRAYS = (\n    \"embeddings_secondary.npy\",\n    \"embeddings_tertiary.npy\",\n    \"hsv_features.npy\",\n)\n\n\ndef _tracklet_stats(tracklet: Any) -> tuple[int, float]:\n    length = int(tracklet.num_frames)\n    avg_confidence = float(tracklet.mean_confidence) if length > 0 else 0.0\n    return length, avg_confidence\n\n\ndef _load_index_map(stage2_dir: Path) -> list[dict[str, Any]]:\n    index_path = stage2_dir / \"embedding_index.json\"\n    if not index_path.exists():\n        raise FileNotFoundError(index_path)\n    payload = json.loads(index_path.read_text(encoding=\"utf-8\"))\n    if not isinstance(payload, list):\n        raise ValueError(f\"Expected list in {index_path}, got {type(payload).__name__}\")\n    return payload\n\n\ndef _copy_filtered_array(input_path: Path, output_path: Path, keep_indices: list[int]) -> list[int]:\n    array = np.load(input_path)\n    if array.shape[0] < len(keep_indices):\n        raise ValueError(\n            f\"Array {input_path} has {array.shape[0]} rows, fewer than selected index count {len(keep_indices)}\"\n        )\n    filtered = array[np.asarray(keep_indices, dtype=np.int64)]\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    np.save(output_path, filtered.astype(array.dtype, copy=False))\n    return [int(value) for value in filtered.shape]\n\n\ndef _copy_filtered_multi_query(input_path: Path, output_path: Path, keep_indices: list[int]) -> list[int] | None:\n    if not input_path.exists():\n        return None\n    with np.load(input_path) as data:\n        if \"embeddings\" not in data:\n            raise KeyError(f\"{input_path} does not contain an 'embeddings' array\")\n        embeddings = data[\"embeddings\"]\n    filtered = embeddings[np.asarray(keep_indices, dtype=np.int64)]\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    np.savez_compressed(output_path, embeddings=filtered.astype(embeddings.dtype, copy=False))\n    return [int(value) for value in filtered.shape]\n\n\ndef filter_stage_outputs(\n    stage1_dir: str | Path,\n    stage2_dir: str | Path,\n    output_stage1_dir: str | Path,\n    output_stage2_dir: str | Path,\n    min_avg_confidence: float,\n    min_length: int,\n    summary_path: str | Path | None = None,\n) -> dict[str, Any]:\n    \"\"\"Write filtered Stage 1/2 artifacts while preserving Stage 2 row order.\"\"\"\n    stage1_dir = Path(stage1_dir)\n    stage2_dir = Path(stage2_dir)\n    output_stage1_dir = Path(output_stage1_dir)\n    output_stage2_dir = Path(output_stage2_dir)\n\n    tracklets_by_camera = load_tracklets_by_camera(stage1_dir)\n    tracklet_lookup = {\n        (camera_id, tracklet.track_id): tracklet\n        for camera_id, tracklets in tracklets_by_camera.items()\n        for tracklet in tracklets\n    }\n    index_map = _load_index_map(stage2_dir)\n\n    keep_indices: list[int] = []\n    filtered_index_map: list[dict[str, Any]] = []\n    filtered_tracklets_by_camera: dict[str, list[Any]] = {camera_id: [] for camera_id in tracklets_by_camera}\n    drop_counts = {\n        \"length_only\": 0,\n        \"confidence_only\": 0,\n        \"both\": 0,\n        \"missing_tracklet\": 0,\n    }\n    per_tracklet: list[dict[str, Any]] = []\n\n    for row_index, row in enumerate(index_map):\n        camera_id = str(row[\"camera_id\"])\n        track_id = int(row[\"track_id\"])\n        tracklet = tracklet_lookup.get((camera_id, track_id))\n        if tracklet is None:\n            drop_counts[\"missing_tracklet\"] += 1\n            per_tracklet.append({\n                \"row_index\": row_index,\n                \"camera_id\": camera_id,\n                \"track_id\": track_id,\n                \"keep\": False,\n                \"drop_reason\": \"missing_tracklet\",\n            })\n            continue\n\n        length, avg_confidence = _tracklet_stats(tracklet)\n        drop_length = length < int(min_length)\n        drop_confidence = avg_confidence < float(min_avg_confidence)\n        keep = not (drop_length or drop_confidence)\n        if keep:\n            keep_indices.append(row_index)\n            filtered_index_map.append(row)\n            filtered_tracklets_by_camera.setdefault(camera_id, []).append(tracklet)\n            drop_reason = None\n        elif drop_length and drop_confidence:\n            drop_counts[\"both\"] += 1\n            drop_reason = \"both\"\n        elif drop_length:\n            drop_counts[\"length_only\"] += 1\n            drop_reason = \"length_only\"\n        else:\n            drop_counts[\"confidence_only\"] += 1\n            drop_reason = \"confidence_only\"\n\n        per_tracklet.append({\n            \"row_index\": row_index,\n            \"camera_id\": camera_id,\n            \"track_id\": track_id,\n            \"length\": length,\n            \"avg_confidence\": avg_confidence,\n            \"keep\": keep,\n            \"drop_reason\": drop_reason,\n        })\n\n    output_stage1_dir.mkdir(parents=True, exist_ok=True)\n    output_stage2_dir.mkdir(parents=True, exist_ok=True)\n    save_tracklets_by_camera(filtered_tracklets_by_camera, output_stage1_dir)\n    (output_stage2_dir / \"embedding_index.json\").write_text(\n        json.dumps(filtered_index_map, indent=2),\n        encoding=\"utf-8\",\n    )\n\n    array_shapes = {\n        \"embeddings.npy\": _copy_filtered_array(stage2_dir / \"embeddings.npy\", output_stage2_dir / \"embeddings.npy\", keep_indices),\n    }\n    for filename in OPTIONAL_STAGE2_ARRAYS:\n        source_path = stage2_dir / filename\n        if source_path.exists():\n            array_shapes[filename] = _copy_filtered_array(source_path, output_stage2_dir / filename, keep_indices)\n    multi_query_shape = _copy_filtered_multi_query(\n        stage2_dir / \"multi_query_embeddings.npz\",\n        output_stage2_dir / \"multi_query_embeddings.npz\",\n        keep_indices,\n    )\n    if multi_query_shape is not None:\n        array_shapes[\"multi_query_embeddings.npz\"] = multi_query_shape\n\n    input_counts_by_camera = {camera_id: len(tracklets) for camera_id, tracklets in tracklets_by_camera.items()}\n    output_counts_by_camera = {\n        camera_id: len(filtered_tracklets_by_camera.get(camera_id, []))\n        for camera_id in input_counts_by_camera\n    }\n    count_delta_by_camera = {\n        camera_id: output_counts_by_camera[camera_id] - input_count\n        for camera_id, input_count in input_counts_by_camera.items()\n    }\n    summary = {\n        \"min_avg_confidence\": float(min_avg_confidence),\n        \"min_length\": int(min_length),\n        \"total_in\": len(index_map),\n        \"total_out\": len(keep_indices),\n        \"total_dropped\": len(index_map) - len(keep_indices),\n        \"drop_counts\": drop_counts,\n        \"input_counts_by_camera\": input_counts_by_camera,\n        \"output_counts_by_camera\": output_counts_by_camera,\n        \"count_delta_by_camera\": count_delta_by_camera,\n        \"keep_indices\": keep_indices,\n        \"array_shapes\": array_shapes,\n        \"per_tracklet\": per_tracklet,\n        \"paths\": {\n            \"stage1_dir\": str(stage1_dir),\n            \"stage2_dir\": str(stage2_dir),\n            \"output_stage1_dir\": str(output_stage1_dir),\n            \"output_stage2_dir\": str(output_stage2_dir),\n        },\n    }\n\n    if summary_path is not None:\n        summary_path = Path(summary_path)\n        summary_path.parent.mkdir(parents=True, exist_ok=True)\n        summary_path.write_text(json.dumps(summary, indent=2), encoding=\"utf-8\")\n\n    return summary\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\"--stage1-dir\", type=Path, required=True)\n    parser.add_argument(\"--stage2-dir\", type=Path, required=True)\n    parser.add_argument(\"--output-stage1-dir\", type=Path, required=True)\n    parser.add_argument(\"--output-stage2-dir\", type=Path, required=True)\n    parser.add_argument(\"--min-avg-confidence\", type=float, required=True)\n    parser.add_argument(\"--min-length\", type=int, required=True)\n    parser.add_argument(\"--summary\", type=Path, default=None)\n    return parser.parse_args()\n\n\ndef main() -> None:\n    args = parse_args()\n    summary = filter_stage_outputs(\n        stage1_dir=args.stage1_dir,\n        stage2_dir=args.stage2_dir,\n        output_stage1_dir=args.output_stage1_dir,\n        output_stage2_dir=args.output_stage2_dir,\n        min_avg_confidence=args.min_avg_confidence,\n        min_length=args.min_length,\n        summary_path=args.summary,\n    )\n    print(json.dumps(summary, indent=2))\n\n\nif __name__ == \"__main__\":\n    main()",
}
for rel_path, content in LOCAL_14I_FILES.items():
    target_path = PROJECT / rel_path
    target_path.parent.mkdir(parents=True, exist_ok=True)
    target_path.write_text(content, encoding="utf-8")
print("Injected 14i filter helper into cloned repo")


def pip(*args: str) -> None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])


pip("numpy", "scipy", "pandas", "faiss-cpu", "motmetrics", "omegaconf", "rich", "networkx>=3.1", "click", "loguru", "scikit-learn")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"], cwd=str(PROJECT))
print(f"Repo ready at {PROJECT}")

## 2. Mount 14h Checkpoint And CityFlowV2 Ground Truth

In [ ]:
SOURCE_14H_OWNER_SLUG = "yahiaakhalafallah/14h-robust-tracklet-pooling"
SOURCE_14H_SLUG = SOURCE_14H_OWNER_SLUG.split("/", 1)[1]
EXPECTED_CAMS = ["S01_c001", "S01_c002", "S01_c003", "S02_c006", "S02_c007", "S02_c008"]


def find_input_dir(slug: str, owner_slug: str, hints=()) -> Path:
    direct = INPUT_ROOT / slug
    if direct.exists():
        return direct

    owner, _, kernel = owner_slug.partition("/")
    nested = INPUT_ROOT / "notebooks" / owner / kernel
    if nested.exists():
        return nested

    lowered_slug = slug.lower()
    lowered_hints = tuple(str(hint).lower() for hint in hints)
    for path in list(INPUT_ROOT.iterdir()) if INPUT_ROOT.exists() else []:
        if not path.is_dir():
            continue
        name = path.name.lower()
        if lowered_slug in name or all(hint in name for hint in lowered_hints):
            return path

    for path in INPUT_ROOT.rglob("checkpoint.tar.gz") if INPUT_ROOT.exists() else []:
        parent_text = str(path.parent).lower()
        if lowered_slug in parent_text or all(hint in parent_text for hint in lowered_hints):
            return path.parent

    return direct


def find_14h_checkpoint() -> Path:
    source_dir = find_input_dir(SOURCE_14H_SLUG, SOURCE_14H_OWNER_SLUG, hints=("14h", "robust", "tracklet"))
    checkpoint_path = source_dir / "checkpoint.tar.gz"
    if checkpoint_path.exists():
        print(f"14h input: {source_dir}")
        return checkpoint_path

    visible = [str(path) for path in INPUT_ROOT.rglob("checkpoint.tar.gz")] if INPUT_ROOT.exists() else []
    raise FileNotFoundError(
        f"14h checkpoint.tar.gz not found for {SOURCE_14H_OWNER_SLUG}. "
        f"Visible checkpoints under /kaggle/input: {visible[:20]}"
    )


checkpoint = find_14h_checkpoint()
EXTRACT_DIR = Path("/tmp/14h_checkpoint")
if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Extracting {checkpoint} ({checkpoint.stat().st_size / 1024**2:.1f} MB)")
with tarfile.open(str(checkpoint), "r:gz") as archive:
    archive.extractall(str(EXTRACT_DIR))

with open(EXTRACT_DIR / "run_metadata.json", encoding="utf-8") as handle:
    previous_meta = json.load(handle)
SOURCE_14H_RUN_NAME = previous_meta["run_name"]
SOURCE_14H_RUN_DIR = EXTRACT_DIR / SOURCE_14H_RUN_NAME
SOURCE_STAGE1_DIR = SOURCE_14H_RUN_DIR / "stage1"
SOURCE_STAGE2_DIR = SOURCE_14H_RUN_DIR / "stage2"
for required_path in [
    SOURCE_STAGE1_DIR,
    SOURCE_STAGE2_DIR / "embeddings.npy",
    SOURCE_STAGE2_DIR / "embeddings_tertiary.npy",
    SOURCE_STAGE2_DIR / "hsv_features.npy",
    SOURCE_STAGE2_DIR / "embedding_index.json",
]:
    if not required_path.exists():
        raise FileNotFoundError(required_path)
print(f"Loaded 14h run: {SOURCE_14H_RUN_NAME}")

def is_cityflow_gt_root(path: Path) -> bool:
    return path.exists() and all((path / cam / "gt" / "gt.txt").exists() for cam in EXPECTED_CAMS)


def find_cityflow_gt_root() -> Path:
    candidates = [
        PROJECT / "data" / "raw" / "cityflowv2",
        EXTRACT_DIR / "gt_annotations",
        Path("/kaggle/input/data-aicity-2023-track-2"),
        Path("/kaggle/input/datasets/thanhnguyenle/data-aicity-2023-track-2"),
    ]
    for candidate in candidates:
        if is_cityflow_gt_root(candidate):
            return candidate

    for gt_file in INPUT_ROOT.rglob("gt.txt") if INPUT_ROOT.exists() else []:
        if gt_file.parent.name != "gt" or gt_file.parent.parent.name not in EXPECTED_CAMS:
            continue
        candidate = gt_file.parents[2]
        if is_cityflow_gt_root(candidate):
            return candidate

    visible = [str(path) for path in INPUT_ROOT.rglob("gt.txt")] if INPUT_ROOT.exists() else []
    raise FileNotFoundError(
        "CityFlowV2 ground-truth annotations not found in normalized camera layout. "
        f"Expected all {EXPECTED_CAMS} under <root>/<cam>/gt/gt.txt. "
        f"Visible gt.txt examples: {visible[:20]}"
    )


GT_DIR = find_cityflow_gt_root()
print(f"Ground truth root: {GT_DIR}")
print("GT files:")
for cam in EXPECTED_CAMS:
    gt_file = GT_DIR / cam / "gt" / "gt.txt"
    print(f"  {cam}: {gt_file} rows={sum(1 for line in gt_file.open() if line.strip())}")

## 3. Define Sweep And Load Stage 2 Features

In [ ]:
from scripts.filter_tracklets import filter_stage_outputs
from src.core.config import load_config, save_config
from src.core.data_models import TrackletFeatures
from src.core.io_utils import load_tracklets_by_camera
from src.core.logging_utils import setup_logging
from src.stage3_indexing import run_stage3
from src.stage4_association import run_stage4
from src.stage5_evaluation import run_stage5

RUN_NAME = f"run_14i_track_quality_prefilter_v2_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
RUN_DIR = DATA_OUT / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
setup_logging(level="INFO", log_file=RUN_DIR / "pipeline.log")
print(f"Run: {RUN_NAME}")

ANCHOR_CONFIG = {
    "aqe_k": 2,
    "fic_regularisation": 0.5,
    "similarity_threshold": 0.48,
    "w_tertiary": 0.525,
}
SWEEP_CONFIGS = [
    {"label": "F0", "block": "drift", "min_avg_confidence": 0.0, "min_length": 0, "notes": "No-filter identity drift gate", **ANCHOR_CONFIG},
]
for min_length in [3, 5, 8, 12]:
    for min_avg_confidence in [0.30, 0.35, 0.40, 0.45, 0.50]:
        SWEEP_CONFIGS.append({
            "label": f"F{len(SWEEP_CONFIGS)}",
            "block": "filter_grid",
            "min_avg_confidence": min_avg_confidence,
            "min_length": min_length,
            "notes": f"Drop tracklets with length < {min_length} or avg confidence < {min_avg_confidence:.2f}",
            **ANCHOR_CONFIG,
        })

F0_REPRO_TARGET = 0.77936
F0_REPRO_TOL = 0.001
F0_ID_SWITCH_TARGET = 154
WIN_THRESHOLD = 0.7810
MARGINAL_MIN = 0.7795
NEUTRAL_MIN = 0.7785
SOLVER = "cc"
ALGORITHM = "conflict_free_cc"
LOUVAIN_RES = 0.70
APPEARANCE_WEIGHT = 0.70
HSV_WEIGHT = 0.0
ST_WEIGHT = round(1.0 - APPEARANCE_WEIGHT - HSV_WEIGHT, 4)
BRIDGE_PRUNE = 0.0
MAX_COMP_SIZE = 12
GALLERY_THRESH = 0.48
ORPHAN_MATCH_THRESH = 0.38
INTRA_MERGE = True
INTRA_MERGE_THRESH = 0.80
INTRA_MERGE_GAP = 30
MULTI_QUERY_WEIGHT = 0.00
MTMC_ONLY = False


def load_metrics(report_path: Path) -> dict:
    if not report_path.exists():
        return {}
    payload = json.loads(report_path.read_text(encoding="utf-8"))
    details = payload.get("details", {}) or {}
    error_analysis = details.get("error_analysis", {}) or {}
    return {
        "mtmc_idf1": payload.get("mtmc_idf1", details.get("mtmc_idf1", payload.get("idf1"))),
        "trackeval_idf1": payload.get("idf1"),
        "mota": payload.get("mota"),
        "hota": payload.get("hota"),
        "id_switches": payload.get("id_switches"),
        "conflations": error_analysis.get("conflated_pred"),
        "fragmentations": error_analysis.get("fragmented_gt"),
        "num_pred_ids": payload.get("num_pred_ids", error_analysis.get("total_pred")),
    }


def summarize_prediction_dir(pred_dir: Path) -> dict:
    files = sorted(pred_dir.glob("*.txt")) if pred_dir.exists() else []
    rows_by_camera = {}
    ids_by_camera = {}
    for pred_file in files:
        rows = [line.strip().split(",") for line in pred_file.open() if line.strip()]
        rows_by_camera[pred_file.stem] = len(rows)
        ids_by_camera[pred_file.stem] = len({row[1] for row in rows if len(row) > 1})
    return {
        "exists": pred_dir.exists(),
        "file_count": len(files),
        "rows_by_camera": rows_by_camera,
        "ids_by_camera": ids_by_camera,
        "total_rows": int(sum(rows_by_camera.values())),
        "total_ids_camera_sum": int(sum(ids_by_camera.values())),
    }


def copy_recovery_artifacts(config_dir: Path, label: str) -> Path:
    recovery_dir = Path("/kaggle/working/outputs/14i_v2_recovery") / label
    if recovery_dir.exists():
        shutil.rmtree(recovery_dir)
    recovery_dir.mkdir(parents=True, exist_ok=True)
    for rel_path in [
        Path("config.yaml"),
        Path("filter_summary.json"),
        Path("stage4/global_trajectories.json"),
        Path("stage4/forensic_report.json"),
        Path("stage5/evaluation_report.json"),
    ]:
        source_path = config_dir / rel_path
        if source_path.exists():
            target_path = recovery_dir / rel_path
            target_path.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source_path, target_path)
    pred_dir = config_dir / "stage5" / "predictions_mot"
    if pred_dir.exists():
        shutil.copytree(pred_dir, recovery_dir / "stage5" / "predictions_mot")
    return recovery_dir


def build_features(stage2_dir: Path) -> tuple[list[TrackletFeatures], dict]:
    index_map = json.loads((stage2_dir / "embedding_index.json").read_text(encoding="utf-8"))
    embeddings = np.load(stage2_dir / "embeddings.npy").astype(np.float32)
    hsv_features = np.load(stage2_dir / "hsv_features.npy").astype(np.float32)
    if embeddings.shape[0] != len(index_map) or hsv_features.shape[0] != len(index_map):
        raise ValueError(
            f"Stage2 row mismatch: embeddings={embeddings.shape}, hsv={hsv_features.shape}, index={len(index_map)}"
        )
    features = [
        TrackletFeatures(
            track_id=int(row["track_id"]),
            camera_id=str(row["camera_id"]),
            class_id=int(row["class_id"]),
            embedding=embeddings[row_index],
            hsv_histogram=hsv_features[row_index],
            raw_embedding=None,
            multi_query_embeddings=None,
        )
        for row_index, row in enumerate(index_map)
    ]
    return features, {
        "num_features": len(features),
        "primary_shape": list(embeddings.shape),
        "hsv_shape": list(hsv_features.shape),
    }

## 4. Run Filter Sweep

In [ ]:
def build_overrides(config: dict, config_run_name: str, stage2_dir: Path) -> list[str]:
    w_tertiary = float(config["w_tertiary"])
    sim_thresh = float(config["similarity_threshold"])
    aqe_k = int(config["aqe_k"])
    fic_reg = float(config["fic_regularisation"])
    return [
        f"project.run_name={config_run_name}",
        f"project.output_dir={DATA_OUT}",
        "stage0.cameras=[S01_c001,S01_c002,S01_c003,S02_c006,S02_c007,S02_c008]",
        f"stage4.association.query_expansion.k={aqe_k}",
        "stage4.association.query_expansion.alpha=5.0",
        "stage4.association.query_expansion.dba=false",
        f"stage4.association.graph.similarity_threshold={sim_thresh}",
        f"stage4.association.solver={SOLVER}",
        f"stage4.association.graph.algorithm={ALGORITHM}",
        f"stage4.association.graph.louvain_resolution={LOUVAIN_RES}",
        f"stage4.association.graph.bridge_prune_margin={BRIDGE_PRUNE}",
        f"stage4.association.graph.max_component_size={MAX_COMP_SIZE}",
        f"stage4.association.weights.vehicle.appearance={APPEARANCE_WEIGHT}",
        f"stage4.association.weights.vehicle.hsv={HSV_WEIGHT}",
        f"stage4.association.weights.vehicle.spatiotemporal={ST_WEIGHT}",
        "stage4.association.mutual_nn.top_k_per_query=20",
        "stage4.association.fic.enabled=true",
        f"stage4.association.fic.regularisation={fic_reg}",
        "stage4.association.reranking.enabled=false",
        "stage4.association.camera_pair_norm.enabled=false",
        "stage4.association.fac.enabled=false",
        f"stage4.association.multi_query.enabled={str(MULTI_QUERY_WEIGHT > 0.0).lower()}",
        f"stage4.association.multi_query.weight={MULTI_QUERY_WEIGHT}",
        f"stage4.association.multi_query.dir={stage2_dir}",
        "stage4.association.aflink.enabled=false",
        "stage4.association.secondary_embeddings.path=",
        "stage4.association.secondary_embeddings.weight=0.0",
        f"stage4.association.tertiary_embeddings.path={stage2_dir / 'embeddings_tertiary.npy'}",
        f"stage4.association.tertiary_embeddings.weight={w_tertiary}",
        "stage4.association.camera_bias.enabled=false",
        "stage4.association.zone_model.enabled=false",
        "stage4.association.hierarchical.enabled=false",
        f"stage4.association.intra_camera_merge.enabled={str(INTRA_MERGE).lower()}",
        f"stage4.association.intra_camera_merge.threshold={INTRA_MERGE_THRESH}",
        f"stage4.association.intra_camera_merge.max_time_gap={INTRA_MERGE_GAP}",
        "stage4.association.gallery_expansion.enabled=true",
        f"stage4.association.gallery_expansion.threshold={GALLERY_THRESH}",
        f"stage4.association.gallery_expansion.orphan_match_threshold={ORPHAN_MATCH_THRESH}",
        "stage4.association.weights.length_weight_power=0.3",
        "stage4.association.temporal_overlap.enabled=true",
        "stage4.association.temporal_overlap.bonus=0.05",
        "stage4.association.temporal_overlap.max_mean_time=5.0",
        f"stage5.mtmc_only_submission={str(MTMC_ONLY).lower()}",
        "stage5.stationary_filter.enabled=true",
        "stage5.stationary_filter.min_displacement_px=150",
        "stage5.stationary_filter.max_mean_velocity_px=2.0",
        "stage5.min_submission_confidence=0.15",
        "stage5.cross_id_nms_iou=0.40",
        "stage5.min_trajectory_confidence=0.30",
        "stage5.min_trajectory_frames=40",
        "stage5.track_edge_trim.enabled=false",
        "stage5.track_smoothing.enabled=false",
        "stage5.gt_frame_clip=true",
        "stage5.gt_zone_filter=true",
        f"stage5.ground_truth_dir={GT_DIR}",
    ]


def run_config(config: dict) -> dict:
    label = config["label"]
    config_dir = RUN_DIR / label
    filtered_stage1_dir = config_dir / "stage1"
    filtered_stage2_dir = config_dir / "stage2"
    filter_summary_path = config_dir / "filter_summary.json"
    config_dir.mkdir(parents=True, exist_ok=True)
    print("\n" + "=" * 80)
    print(
        f"Running {label}: tau_c={config['min_avg_confidence']:.2f}, L_min={config['min_length']}, "
        f"w_tertiary={config['w_tertiary']:.3f}, sim_thresh={config['similarity_threshold']:.2f}"
    )
    print("=" * 80)

    filter_summary = filter_stage_outputs(
        stage1_dir=SOURCE_STAGE1_DIR,
        stage2_dir=SOURCE_STAGE2_DIR,
        output_stage1_dir=filtered_stage1_dir,
        output_stage2_dir=filtered_stage2_dir,
        min_avg_confidence=float(config["min_avg_confidence"]),
        min_length=int(config["min_length"]),
        summary_path=filter_summary_path,
    )
    tracklets_by_camera = load_tracklets_by_camera(filtered_stage1_dir)
    features, feature_summary = build_features(filtered_stage2_dir)
    print(
        f"Filter kept {filter_summary['total_out']}/{filter_summary['total_in']} tracklets; "
        f"dropped={filter_summary['total_dropped']}"
    )

    config_run_name = f"{RUN_NAME}_{label}"
    cfg = load_config(
        "configs/default.yaml",
        dataset_config="configs/datasets/cityflowv2.yaml",
        overrides=build_overrides(config, config_run_name, filtered_stage2_dir),
    )
    save_config(cfg, config_dir / "config.yaml")

    start = time.time()
    faiss_index, metadata_store = run_stage3(cfg, features, tracklets_by_camera, output_dir=config_dir / "stage3")
    stage3_min = (time.time() - start) / 60.0
    print(f"{label} Stage 3 complete in {stage3_min:.2f} min")

    start = time.time()
    trajectories = run_stage4(cfg, faiss_index, metadata_store, features, tracklets_by_camera, output_dir=config_dir / "stage4")
    stage4_min = (time.time() - start) / 60.0
    print(f"{label} Stage 4 complete in {stage4_min:.2f} min: {len(trajectories)} global trajectories")

    start = time.time()
    run_stage5(cfg, trajectories, output_dir=config_dir / "stage5")
    stage5_min = (time.time() - start) / 60.0
    print(f"{label} Stage 5 complete in {stage5_min:.2f} min")

    report_path = config_dir / "stage5" / "evaluation_report.json"
    metrics = load_metrics(report_path)
    prediction_summary = summarize_prediction_dir(config_dir / "stage5" / "predictions_mot")
    recovery_dir = copy_recovery_artifacts(config_dir, label)
    idf1_value = metrics.get("mtmc_idf1")
    if idf1_value is None:
        idf1_value = metrics.get("trackeval_idf1")
    if idf1_value is None:
        raise RuntimeError(f"IDF1 not found in {report_path}")
    if not prediction_summary["exists"] or prediction_summary["file_count"] == 0:
        raise RuntimeError(f"No MOT prediction files were written for {label}: {prediction_summary}")
    if idf1_value == 0.0 and prediction_summary["total_rows"] == 0:
        raise RuntimeError(f"Zero IDF1 with zero prediction rows for {label}: {prediction_summary}")

    row = {
        "label": label,
        "block": config["block"],
        "min_avg_confidence": float(config["min_avg_confidence"]),
        "min_length": int(config["min_length"]),
        "w_tertiary": float(config["w_tertiary"]),
        "w_primary": round(1.0 - float(config["w_tertiary"]), 6),
        "w_secondary": 0.0,
        "similarity_threshold": float(config["similarity_threshold"]),
        "aqe_k": int(config["aqe_k"]),
        "fic_regularisation": float(config["fic_regularisation"]),
        "notes": config["notes"],
        "idf1": idf1_value,
        "mtmc_idf1": metrics.get("mtmc_idf1"),
        "trackeval_idf1": metrics.get("trackeval_idf1"),
        "mota": metrics.get("mota"),
        "hota": metrics.get("hota"),
        "id_switches": metrics.get("id_switches"),
        "conflations": metrics.get("conflations"),
        "fragmentations": metrics.get("fragmentations"),
        "num_pred_ids": metrics.get("num_pred_ids"),
        "num_trajectories": len(trajectories),
        "num_stage4_tracklets": int(sum(len(trajectory.tracklets) for trajectory in trajectories)),
        "prediction_summary": prediction_summary,
        "filter_summary": {
            key: filter_summary[key]
            for key in [
                "total_in",
                "total_out",
                "total_dropped",
                "drop_counts",
                "input_counts_by_camera",
                "output_counts_by_camera",
                "count_delta_by_camera",
                "array_shapes",
            ]
        },
        "feature_summary": feature_summary,
        "stage_timings_min": {
            "filter": 0.0,
            "stage3": round(stage3_min, 2),
            "stage4": round(stage4_min, 2),
            "stage5": round(stage5_min, 2),
        },
        "paths": {
            "config_dir": str(config_dir),
            "filtered_stage1_dir": str(filtered_stage1_dir),
            "filtered_stage2_dir": str(filtered_stage2_dir),
            "filter_summary": str(filter_summary_path),
            "evaluation_report": str(report_path),
            "recovery_dir": str(recovery_dir),
        },
    }
    print(f"{label} MTMC IDF1: {idf1_value:.5f} id_switches={row['id_switches']}")
    return row


results = []
filter_results = []
halt_reason = None
drift_detected = False
wall_start = time.time()

drift_check_result = run_config(SWEEP_CONFIGS[0])
results.append(drift_check_result)
(RUN_DIR / "14i_partial_results.json").write_text(json.dumps(results, indent=2), encoding="utf-8")

f0_idf1 = float(drift_check_result["idf1"])
f0_id_switches = drift_check_result.get("id_switches")
if abs(f0_idf1 - F0_REPRO_TARGET) > F0_REPRO_TOL or f0_id_switches != F0_ID_SWITCH_TARGET:
    drift_detected = True
    halt_reason = (
        f"F0 drift gate failed: got idf1={f0_idf1:.5f}, id_switches={f0_id_switches}; "
        f"expected idf1 within {F0_REPRO_TARGET:.5f} +/- {F0_REPRO_TOL:.5f} "
        f"and id_switches={F0_ID_SWITCH_TARGET}"
    )
    print(halt_reason)
else:
    print(f"F0 drift gate passed: {f0_idf1:.5f}, id_switches={f0_id_switches}")
    for config in SWEEP_CONFIGS[1:]:
        row = run_config(config)
        results.append(row)
        filter_results.append(row)
        (RUN_DIR / "14i_partial_results.json").write_text(json.dumps(results, indent=2), encoding="utf-8")

overall_best = max(results, key=lambda row: row["idf1"] if row["idf1"] is not None else -1.0)
best_filter = max(filter_results, key=lambda row: row["idf1"] if row["idf1"] is not None else -1.0) if filter_results else None

if drift_detected:
    verdict = "DRIFT"
elif float(overall_best["idf1"]) >= WIN_THRESHOLD:
    verdict = "WIN"
elif float(overall_best["idf1"]) >= MARGINAL_MIN:
    verdict = "MARGINAL"
elif float(overall_best["idf1"]) >= NEUTRAL_MIN:
    verdict = "NEUTRAL"
else:
    verdict = "DEAD"

promoted_config = overall_best if verdict == "WIN" and not drift_detected else None
print("\n" + "#" * 80)
print(
    f"BEST 14i CONFIG: {overall_best['label']} tau_c={overall_best['min_avg_confidence']:.2f} "
    f"L_min={overall_best['min_length']} MTMC_IDF1={overall_best['idf1']:.5f} "
    f"id_switches={overall_best.get('id_switches')} verdict={verdict} drift={drift_detected}"
)
if halt_reason:
    print(f"HALT: {halt_reason}")
print("#" * 80)

## 5. Save 14i Summary

In [ ]:
summary_dir = Path("/kaggle/working/outputs/14i_v2_summary")
summary_dir.mkdir(parents=True, exist_ok=True)
summary_path = summary_dir / "14i_summary.json"
root_summary_path = Path("/kaggle/working/14i_summary.json")

summary = {
    "run_name": RUN_NAME,
    "source_14h_run_name": SOURCE_14H_RUN_NAME,
    "experiment": "14i-track-quality-prefilter",
    "kernel": "yahiaakhalafallah/14i-track-quality-prefilter",
    "frame_id_convention": "0-based internal Stage 1/2 frame IDs; MOT output is converted to 1-based in Stage 5",
    "drift_detected": drift_detected,
    "drift_check_label": "F0",
    "drift_check_result": drift_check_result,
    "drift_reproduction_target": F0_REPRO_TARGET,
    "drift_reproduction_tolerance": F0_REPRO_TOL,
    "drift_id_switch_target": F0_ID_SWITCH_TARGET,
    "promoted_config": promoted_config,
    "verdict": verdict,
    "filter_grid": {
        "min_lengths": [3, 5, 8, 12],
        "min_avg_confidences": [0.30, 0.35, 0.40, 0.45, 0.50],
        "grid_size": 20,
    },
    "mini_sweep": {
        "description": "No-filter drift gate plus track-quality pre-filter grid at the 14e B1 anchor",
        "grid": SWEEP_CONFIGS,
        "execution_order": [row["label"] for row in results],
        "results": results,
        "best": overall_best,
        "best_filter": best_filter,
        "grid_size": len(SWEEP_CONFIGS),
        "halt_reason": halt_reason,
    },
    "results": results,
    "filter_results": filter_results,
    "overall_best": overall_best,
    "best": overall_best,
    "best_filter": best_filter,
    "halt_reason": halt_reason,
    "total_config_count": len(SWEEP_CONFIGS),
    "executed_config_count": len(results),
    "feature_sources": {
        "source_kernel": SOURCE_14H_OWNER_SLUG,
        "checkpoint": str(checkpoint),
        "stage1_tracklets": str(SOURCE_STAGE1_DIR),
        "stage2_features": str(SOURCE_STAGE2_DIR),
    },
    "fixed_config": {
        "pca_components": 384,
        "algorithm": ALGORITHM,
        "aqe_k": 2,
        "fic_regularisation": 0.5,
        "similarity_threshold": 0.48,
        "w_primary": 0.475,
        "w_secondary": 0.0,
        "w_tertiary": 0.525,
        "gallery_expansion": True,
        "gallery_threshold": GALLERY_THRESH,
        "orphan_match_threshold": ORPHAN_MATCH_THRESH,
        "temporal_overlap_bonus": 0.05,
        "intra_merge": INTRA_MERGE,
        "intra_merge_threshold": INTRA_MERGE_THRESH,
        "intra_merge_gap": INTRA_MERGE_GAP,
        "mtmc_only_submission": MTMC_ONLY,
    },
    "stop_criteria": {
        "win_threshold": WIN_THRESHOLD,
        "marginal_min": MARGINAL_MIN,
        "neutral_min": NEUTRAL_MIN,
        "dead_below": NEUTRAL_MIN,
        "drift_condition": f"abs(F0 - {F0_REPRO_TARGET}) > {F0_REPRO_TOL} or id_switches != {F0_ID_SWITCH_TARGET}",
    },
    "stage_timings_min": {
        "stage345_sweep": round((time.time() - wall_start) / 60.0, 2),
    },
    "paths": {
        "run_dir": str(RUN_DIR),
        "summary": str(summary_path),
        "root_summary": str(root_summary_path),
    },
}

summary_payload = json.dumps(summary, indent=2)
summary_path.write_text(summary_payload, encoding="utf-8")
root_summary_path.write_text(summary_payload, encoding="utf-8")
print(f"Saved summary: {summary_path}")
print(json.dumps(summary, indent=2))